# Person / Fire / Smoke — Training Pipeline (Google Colab)

Runs the pipeline from `training.zip`: prep COCO-person + D-Fire → merge → fine-tune YOLOv8 → evaluate → export ONNX.

**Before you start:** in Colab, go to `Runtime -> Change runtime type -> T4 GPU` (or better), then run the cells top to bottom.

In [ ]:
# 0. Confirm a GPU is attached
!nvidia-smi

## 1. Get `training.zip` onto the Colab machine

Option A — upload directly (fine for one-off runs, but you'll re-upload if the runtime disconnects):

In [ ]:
from google.colab import files
uploaded = files.upload()  # select training.zip in the dialog

Option B (recommended) — put `training.zip` in your Google Drive first, then mount Drive so downloads/checkpoints survive disconnects:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Adjust this path to wherever you put the zip in Drive:
DRIVE_ZIP = '/content/drive/MyDrive/training.zip'
import shutil, os
if os.path.exists(DRIVE_ZIP):
    shutil.copy(DRIVE_ZIP, '/content/training.zip')
    print('Copied from Drive.')
else:
    print('Not found in Drive — use the upload cell above instead.')

In [ ]:
# 2. Unzip
%cd /content
!unzip -oq training.zip
%cd training
!ls

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt

## 4. Check the config before running anything

Open `configs/train_config.yaml` in the Colab file browser (left sidebar) and check:

- `dfire.source_class_map` — **must match** the actual label convention in the cloned D-Fire repo (verify after step 5b, before merging).
- `coco.max_samples_per_split` — 8000/split is heavy for a free Colab session; consider lowering (e.g. 1500–3000) for a faster first run.
- `train.epochs` / `train.batch` — free-tier T4 has ~15GB VRAM; `yolov8n.pt` at batch 16 should fit, but drop batch size if you switch to `yolov8s.pt`/`yolov9s.pt` or hit OOM errors.
- `train.device` — already set to `0`, which is correct for a single Colab GPU.

In [ ]:
# 5a. Pull the person-only COCO subset
!python data/prepare_coco_person.py

In [ ]:
# 5b. Clone + remap D-Fire
# After this finishes, double check configs/train_config.yaml -> dfire.source_class_map
# against the cloned repo's own classes.txt / data.yaml before continuing.
!python data/prepare_dfire.py

In [ ]:
# 6. Merge into one YOLO dataset (train/val split + data.yaml)
!python data/merge_datasets.py

In [ ]:
# 7. Fine-tune
# This is the long step. If Colab disconnects mid-run, re-running train.py
# will start a new run under training/runs/ (Ultralytics handles resume
# separately — see its docs if you need to resume from a checkpoint).
!python train.py

In [ ]:
# 8. Evaluate — check per-class mAP (fire/smoke will likely trail person)
!python evaluate.py

In [ ]:
# 9. Export to ONNX (copied to ../config/model/custom_model.onnx)
!python export.py

## 10. Get your results out of Colab

Colab's disk is ephemeral — copy the exported model (and ideally the whole `runs/` dir, for the training logs/checkpoints) somewhere durable before the session ends.

In [ ]:
# Option A: download the ONNX model directly
from google.colab import files
files.download('/content/config/model/custom_model.onnx')

In [ ]:
# Option B: copy everything (model + training runs) back to Drive
import shutil, os
os.makedirs('/content/drive/MyDrive/training_output', exist_ok=True)
shutil.copy('/content/config/model/custom_model.onnx', '/content/drive/MyDrive/training_output/')
shutil.copytree('/content/training/runs', '/content/drive/MyDrive/training_output/runs', dirs_exist_ok=True)
print('Copied to Google Drive: MyDrive/training_output/')